In [1]:
"""
Cement Embodied Carbon Calculator - cradle to gate, EN 15804 modules A1-A3.

Functional unit: 1 tonne cement.
All results in kg CO2e per tonne of cement unless stated otherwise.

A1 emission: CO2 from raw material extraction
A2 emission: CO2 from transport
A3 emission: CO2 from manufacturing

A1 contains 1) quarrying raw materials, 2) crushing, 3) preparing raw meal
A2 contains transport of raw materials, gypsum, SCM and fuel to the plant
A3 contains 1) kiln fuel combustion, 2) electricity, 3) calcination

Standard library only - no pandas, no numpy, no argparse.
Runs in a Jupyter notebook and from a terminal.

    main()                  # ask for every parameter, then print the report

Press Enter at any question, or type 'na', to accept the default in brackets.
"""


"\nCement Embodied Carbon Calculator - cradle to gate, EN 15804 modules A1-A3.\n\nFunctional unit: 1 tonne cement.\nAll results in kg CO2e per tonne of cement unless stated otherwise.\n\nA1 emission: CO2 from raw material extraction\nA2 emission: CO2 from transport\nA3 emission: CO2 from manufacturing\n\nA1 contains 1) quarrying raw materials, 2) crushing, 3) preparing raw meal\nA2 contains transport of raw materials, gypsum, SCM and fuel to the plant\nA3 contains 1) kiln fuel combustion, 2) electricity, 3) calcination\n\nStandard library only - no pandas, no numpy, no argparse.\nRuns in a Jupyter notebook and from a terminal.\n\n    main()                  # ask for every parameter, then print the report\n\nPress Enter at any question, or type 'na', to accept the default in brackets.\n"

In [2]:
import json
from dataclasses import asdict, dataclass, field, fields

In [3]:
# ---------------------------------------------------------------------------
# Chemistry
# ---------------------------------------------------------------------------

M_Ca = 40.078   # g/mol
M_O  = 15.999
M_Si = 28.085
M_Al = 26.982
M_Fe = 55.845
M_C  = 12.011

M_CaO   = M_Ca + M_O                # 56.08
M_SiO2  = M_Si + 2 * M_O            # 60.08
M_Al2O3 = 2 * M_Al + 3 * M_O        # 101.96
M_Fe2O3 = 2 * M_Fe + 3 * M_O        # 159.69
M_CO2   = M_C + 2 * M_O             # 44.01
M_CaCO3 = M_CaO + M_CO2             # 100.09

# stoichiometric ratios
CaCO3_PER_CaO = M_CaCO3 / M_CaO     # 1.785
CO2_PER_CaCO3 = M_CO2 / M_CaCO3     # 0.440

# Bogue calculation, clinker phases -> mass fraction of CaO in each phase.
# C3S  = 3CaO.SiO2            (alite)
# C2S  = 2CaO.SiO2            (belite)
# C3A  = 3CaO.Al2O3           (aluminate)
# C4AF = 4CaO.Al2O3.Fe2O3     (ferrite)
CaO_FRACTION = {
    "C3S":  3 * M_CaO / (3 * M_CaO + M_SiO2),
    "C2S":  2 * M_CaO / (2 * M_CaO + M_SiO2),
    "C3A":  3 * M_CaO / (3 * M_CaO + M_Al2O3),
    "C4AF": 4 * M_CaO / (4 * M_CaO + M_Al2O3 + M_Fe2O3),
}
PHASE_NAMES = list(CaO_FRACTION)


In [4]:
# ---------------------------------------------------------------------------
# Emission factors & Cement reference
# ---------------------------------------------------------------------------

# Kiln fuels: net calorific value (MJ/kg) and combustion factor (kg CO2 / kg fuel).
# IPCC EF database;
# do not mix a gross-CV emission factor with a net CV here.
FUELS = {
    "coal":        (25.8, 2.42),
    "petcoke":     (32.5, 3.40),
    "natural gas": (48.0, 2.75),
    "fuel oil":    (40.4, 3.17),
    "diesel":      (43.0, 3.17),   # per kg; 2.68 kg CO2/L is the per-litre figure
}

# GLEC Framework 2019, well-to-wheel, kg CO2e per tonne-km.
# These numbers already allow for typical empty running, so do not double for return legs.
TRANSPORT_EF = {
    "truck":           0.062,
    "rail (electric)": 0.022,
    "rail (diesel)":   0.041,
}

# Cement type -> composition (% by mass of finished cement) and SCM supply factor.
CEMENT_TYPES = {
    "CEM I":      dict(clinker=95.0, gypsum=5.0, scm=0.0,  scm_ef=0.0),
    "CEM II/B-S": dict(clinker=70.0, gypsum=5.0, scm=25.0, scm_ef=79.0),
    "CEM III/A":  dict(clinker=45.0, gypsum=5.0, scm=50.0, scm_ef=79.0),
    }


In [5]:
# ---------------------------------------------------------------------------
# Inputs - every field is asked for at run time, and adjustable in code
# ---------------------------------------------------------------------------

@dataclass
class Inputs:
    """Every parameter needs a TYPE ANNOTATION (`name: float = value`).

    Without one, @dataclass ignores the line: it becomes a shared class
    attribute instead of a per-instance field, disappears from asdict()
    and from_dict(), and the constructor rejects it as a keyword.
    """    

    cement_type: str = "CEM I"

    # -- Cement composition, % by mass of finished cement. Must sum to 100.
    clinker_pct: float = 95.0
    gypsum_pct: float = 5.0
    scm_pct: float = 0.0
    scm_ef: float = 0.0                          # kg CO2e per tonne of SCM

    # -- Clinker phase composition, % by mass of clinker. Must sum to 100.
    # Gypsum is interground after the kiln, so it does NOT belong in this table.
    phase_pct: tuple = (52.6, 26.3, 10.6, 10.5)

    # -- Raw materials
    limestone_purity_pct: float = 95.0
    clay_t_per_t_clinker: float = 0.20
    sand_t_per_t_clinker: float = 0.03
    iron_t_per_t_clinker: float = 0.02

    # -- A1.1 quarrying, per tonne of rock extracted
    quarry_diesel_l_per_t: float = 1.0
    diesel_ef_per_l: float = 2.68                # kg CO2/L
    anfo_kg_per_t: float = 0.35
    anfo_ef: float = 0.26                        # kg CO2/kg, Sapko et al. (2002)
    gypsum_ef: float = 15.0                      # kg CO2e per tonne quarried gypsum

    # -- A1.2 crushing: (throughput t/h, motor rating kW) per crusher stage
    crushers: tuple = ((600.0, 220.0), (400.0, 150.0))

    # -- A2 transport: (mode, one-way distance km) per leg
    raw_material_haul: tuple = ("truck", 50.0)
    gypsum_haul: tuple = ("truck", 200.0)
    scm_haul: tuple = ("rail (electric)", 300.0)
    fuel_haul: tuple = ("rail (diesel)", 300.0)

    # -- A3.1 kiln. Specific heat consumption is per tonne of CLINKER.
    kiln_heat_mj_per_t_clinker: float = 3000.0
    # Share of kiln heat by fuel, %. Must sum to 100.
    fuel_mix: dict = field(default_factory=lambda: {"coal": 70.0, "natural gas": 30.0})

    # -- A3.2 electricity, kWh per tonne of cement
    power_raw_mill: float = 25.0
    power_kiln_and_preheater: float = 25.0
    power_cooler_fans: float = 5.0
    power_cement_mill: float = 40.0
    power_conveyors_and_packing: float = 8.0

    # -- Grid. UK DESNZ 2023 location-based, incl. T&D losses.
    grid_ef: float = 0.233                       # kg CO2e/kWh


In [6]:
 # -- validation ----------------------------------------------------------
    # Defined INSIDE the class. Attaching it afterwards with
    # `Inputs.validate = validate` breaks on a kernel restart.

def validate(self) -> None:
        def check_sum(label, values, target=100.0, tol=0.01):
            total = sum(values)
            if abs(total - target) > tol:
                raise ValueError(f"{label} must sum to {target:g}, got {total:g}")

        check_sum("cement composition (clinker + gypsum + scm)",
                  [self.clinker_pct, self.gypsum_pct, self.scm_pct])
        check_sum("clinker phase composition", self.phase_pct)
        check_sum("kiln fuel mix", self.fuel_mix.values())

        if len(self.phase_pct) != len(PHASE_NAMES):
            raise ValueError(f"phase_pct needs {len(PHASE_NAMES)} values, "
                             f"got {len(self.phase_pct)}")
        if not 0 < self.limestone_purity_pct <= 100:
            raise ValueError("limestone_purity_pct must be in (0, 100]")
        if not self.crushers:
            raise ValueError("at least one crusher stage is required")
        for throughput, power in self.crushers:
            if throughput <= 0 or power <= 0:
                raise ValueError("crusher throughput and power must be positive")
        for fuel in self.fuel_mix:
            if fuel not in FUELS:
                raise ValueError(f"unknown fuel {fuel!r}; choose from {sorted(FUELS)}")
        for mode, distance in (self.raw_material_haul, self.gypsum_haul,
                               self.scm_haul, self.fuel_haul):
            if mode not in TRANSPORT_EF:
                raise ValueError(f"unknown transport mode {mode!r}; "
                                 f"choose from {sorted(TRANSPORT_EF)}")
            if distance < 0:
                raise ValueError("transport distance cannot be negative")

        non_negative = [
            "scm_ef", "clay_t_per_t_clinker", "sand_t_per_t_clinker",
            "iron_t_per_t_clinker", "quarry_diesel_l_per_t", "diesel_ef_per_l",
            "anfo_kg_per_t", "anfo_ef", "gypsum_ef", "kiln_heat_mj_per_t_clinker",
            "power_raw_mill", "power_kiln_and_preheater", "power_cooler_fans",
            "power_cement_mill", "power_conveyors_and_packing", "grid_ef",
        ]
        for name in non_negative:
            if getattr(self, name) < 0:
                raise ValueError(f"{name} cannot be negative")


In [7]:
# ---------------------------------------------------------------------------
# Serialisation
# ---------------------------------------------------------------------------

def to_dict(inp: Inputs) -> dict:
    return asdict(inp)


def from_dict(data: dict) -> Inputs:
    known = {f.name for f in fields(Inputs)}
    unknown = set(data) - known
    if unknown:
        raise ValueError(f"unknown parameter(s) {sorted(unknown)}; "
                         f"valid names are {sorted(known)}")
    inp = Inputs()
    for name, value in data.items():
        # JSON has no tuples, so restore them for fields that expect one
        if isinstance(getattr(inp, name), tuple) and isinstance(value, list):
            value = tuple(tuple(v) if isinstance(v, list) else v for v in value)
        setattr(inp, name, value)
    return inp


def save_config(inp: Inputs, path: str) -> None:
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(to_dict(inp), handle, indent=2)
    print(f"Saved {path}")


def load_config(path: str) -> Inputs:
    with open(path, encoding="utf-8") as handle:
        return from_dict(json.load(handle))


def list_params(inp: Inputs = None) -> None:
    inp = inp or Inputs()
    print("\nParameters and current values\n")
    for name, value in to_dict(inp).items():
        print(f"  {name:<32} {json.dumps(value)}")
    print(f"\n  cement types:    {', '.join(CEMENT_TYPES)}")
    print(f"  kiln fuels:      {', '.join(FUELS)}")
    print(f"  transport modes: {', '.join(TRANSPORT_EF)}\n")


In [8]:
# ---------------------------------------------------------------------------
# Material balance
# ---------------------------------------------------------------------------

def clinker_cao_fraction(phase_pct):
    """Per-phase rows for display, and the CaO mass fraction of clinker.

    Rounds only at print time, never mid-calculation.
    """
    rows = list(zip(PHASE_NAMES, phase_pct, CaO_FRACTION.values()))
    cao_fraction = sum(pct / 100 * frac for _, pct, frac in rows)
    return rows, cao_fraction


def material_balance(inp: Inputs) -> dict:
    """Masses in tonnes per tonne of finished cement."""
    clinker = inp.clinker_pct / 100
    _, cao_fraction = clinker_cao_fraction(inp.phase_pct)

    # All clinker CaO is attributed to carbonate decomposition (IPCC Tier 2).
    caco3 = clinker * cao_fraction * CaCO3_PER_CaO
    limestone = caco3 / (inp.limestone_purity_pct / 100)

    clay = clinker * inp.clay_t_per_t_clinker
    sand = clinker * inp.sand_t_per_t_clinker
    iron = clinker * inp.iron_t_per_t_clinker

    return {
        "cao_fraction": cao_fraction,
        "caco3": caco3,
        "limestone": limestone,
        "clay": clay,
        "sand": sand,
        "iron_ore": iron,
        "quarried_rock": limestone + clay + sand + iron,
        "gypsum": inp.gypsum_pct / 100,
        "scm": inp.scm_pct / 100,
    }


In [9]:
# ---------------------------------------------------------------------------
# Stage A1 - raw material supply
# ---------------------------------------------------------------------------

def stage_a1(inp: Inputs, mb: dict) -> dict:
    rock = mb["quarried_rock"]
    kwh_per_t_rock = sum(power / throughput for throughput, power in inp.crushers)

    return {
        "A1.1 quarrying - mobile plant diesel":
            rock * inp.quarry_diesel_l_per_t * inp.diesel_ef_per_l,
        "A1.1 quarrying - blasting (ANFO)":
            rock * inp.anfo_kg_per_t * inp.anfo_ef,
        "A1.2 crushing and screening":
            rock * kwh_per_t_rock * inp.grid_ef,
        "A1.3 gypsum supply":
            mb["gypsum"] * inp.gypsum_ef,
        "A1.3 SCM supply":
            mb["scm"] * inp.scm_ef,
    }


In [11]:
# ---------------------------------------------------------------------------
# Stage A2 - transport to the plant
# ---------------------------------------------------------------------------

def stage_a2(inp: Inputs, mb: dict, fuel_mass_t: float) -> dict:
    def leg(mass_t, haul):
        mode, distance = haul
        return mass_t * distance * TRANSPORT_EF[mode]

    return {
        "A2 raw materials quarry -> plant": leg(mb["quarried_rock"], inp.raw_material_haul),
        "A2 gypsum supplier -> plant": leg(mb["gypsum"], inp.gypsum_haul),
        "A2 SCM supplier -> plant": leg(mb["scm"], inp.scm_haul),
        "A2 fuel supplier -> plant": leg(fuel_mass_t, inp.fuel_haul),
    }


In [12]:
# ---------------------------------------------------------------------------
# Stage A3 - manufacturing
# ---------------------------------------------------------------------------

def kiln_fuel(inp: Inputs):
    """Kiln combustion CO2 by fuel, and total delivered fuel mass (t/t cement).

    Specific heat consumption is quoted per tonne of clinker, so it must be
    scaled by the clinker factor to reach the per-tonne-of-cement basis.
    """
    heat_mj = inp.kiln_heat_mj_per_t_clinker * inp.clinker_pct / 100

    emissions, total_mass_t = {}, 0.0
    for fuel, share_pct in inp.fuel_mix.items():
        net_cv, ef = FUELS[fuel]
        mass_kg = heat_mj * share_pct / 100 / net_cv
        emissions[f"A3.1 kiln combustion - {fuel} ({share_pct:g}%)"] = mass_kg * ef
        total_mass_t += mass_kg / 1000

    return emissions, total_mass_t


def stage_a3(inp: Inputs, mb: dict):
    combustion, fuel_mass_t = kiln_fuel(inp)

    electricity = {
        "A3.2 raw mill": inp.power_raw_mill,
        "A3.2 kiln drive and preheater fans": inp.power_kiln_and_preheater,
        "A3.2 clinker cooler fans": inp.power_cooler_fans,
        "A3.2 cement (finish) mill": inp.power_cement_mill,
        "A3.2 conveyors, pumps and packing": inp.power_conveyors_and_packing,
    }
    electricity = {k: kwh * inp.grid_ef for k, kwh in electricity.items()}

    # Calcination uses pure CaCO3, not the as-quarried limestone.
    calcination = {"A3.3 calcination of CaCO3": mb["caco3"] * CO2_PER_CaCO3 * 1000}

    return {**combustion, **electricity, **calcination}, fuel_mass_t


In [13]:
# ---------------------------------------------------------------------------
# Calculation and report
# ---------------------------------------------------------------------------

def calculate(inp: Inputs):
    """Returns (line items, stage totals, material balance). Prints nothing."""
    validate(inp)
    mb = material_balance(inp)

    a3, fuel_mass_t = stage_a3(inp, mb)
    items = {**stage_a1(inp, mb), **stage_a2(inp, mb, fuel_mass_t), **a3}

    stages = {}
    for name, value in items.items():
        stage = name.split()[0].split(".")[0]        # "A1.1 ..." -> "A1"
        stages[stage] = stages.get(stage, 0.0) + value

    return items, stages, mb


def report(inp: Inputs) -> float:
    items, stages, mb = calculate(inp)
    phases, cao = clinker_cao_fraction(inp.phase_pct)
    total = sum(stages.values())

    rule = "=" * 70
    print(f"\n{rule}\n{inp.cement_type} - cradle to gate (A1-A3), 1 tonne of cement\n{rule}")

    print("\nClinker phase composition")
    for name, pct, frac in phases:
        print(f"  {name:<8} {pct:>6.1f} %      CaO fraction {frac:.4f}")
    print(f"  {'CaO in clinker':<8} {cao * 100:>6.2f} %")

    print("\nMaterial balance (tonne per tonne of cement)")
    for key in ("caco3", "limestone", "clay", "sand", "iron_ore", "gypsum", "scm"):
        if mb[key]:
            print(f"  {key.replace('_', ' '):<44} {mb[key]:>8.3f}")

    print("\nEmissions by line item (kg CO2e per tonne of cement)")
    for name, value in items.items():
        print(f"  {name:<44} {value:>8.2f}")

    print("\nEmissions by stage")
    for stage in sorted(stages):
        print(f"  {stage:<44} {stages[stage]:>8.2f}   {stages[stage] / total * 100:>5.1f} %")
    print(f"  {'TOTAL A1-A3':<44} {total:>8.2f}   100.0 %")

    print(f"\n{inp.cement_type}: {total:.1f} kg CO2e per tonne of cement.\n")
    return total


In [14]:
# ---------------------------------------------------------------------------
# Saving and loading a scenario
# ---------------------------------------------------------------------------

def save_config(inp: Inputs, path: str) -> None:
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(inp.to_dict(), handle, indent=2)
    print(f"Saved {path}")


def load_config(path: str) -> Inputs:
    with open(path, encoding="utf-8") as handle:
        return Inputs.from_dict(json.load(handle))


def list_params(inp: Inputs = None) -> None:
    inp = inp or Inputs()
    print("\nParameters and current values\n")
    for name, value in inp.to_dict().items():
        print(f"  {name:<32} {json.dumps(value)}")
    print(f"\n  cement types:    {', '.join(CEMENT_TYPES)}")
    print(f"  kiln fuels:      {', '.join(FUELS)}")
    print(f"  transport modes: {', '.join(TRANSPORT_EF)}\n")


In [15]:
# ---------------------------------------------------------------------------
# Prompt helpers - Enter or 'na' accepts the default
# ---------------------------------------------------------------------------

NO_DATA = ("", "na", "n/a", "?", "-")


def ask_float(prompt: str, default: float, lo: float, hi: float) -> float:
    while True:
        raw = input(f"  {prompt} [{default:g}]: ").strip()
        if raw.lower() in NO_DATA:
            return float(default)
        try:
            value = float(raw)
        except ValueError:
            print(f"    '{raw}' is not a number. Enter = use {default:g}.")
            continue
        if not lo <= value <= hi:
            print(f"    Must be between {lo:g} and {hi:g}. Enter = use {default:g}.")
            continue
        return value


def ask_int(prompt: str, default: int, lo: int, hi: int) -> int:
    while True:
        raw = input(f"  {prompt} [{default}]: ").strip()
        if raw.lower() in NO_DATA:
            return default
        try:
            value = int(raw)
        except ValueError:
            print(f"    '{raw}' is not a whole number. Enter = use {default}.")
            continue
        if not lo <= value <= hi:
            print(f"    Must be between {lo} and {hi}. Enter = use {default}.")
            continue
        return value


def ask_choice(prompt: str, options, default: str) -> str:
    listed = ", ".join(options)
    while True:
        raw = input(f"  {prompt} ({listed}) [{default}]: ").strip()
        if raw.lower() in NO_DATA:
            return default
        if raw in options:
            return raw
        print(f"    Not one of: {listed}. Enter = use {default}.")


def ask_composition(label: str, names, defaults) -> tuple:
    """Ask for a set of percentages, normalising to 100 if they do not add up."""
    values = [ask_float(name, d, 0, 100) for name, d in zip(names, defaults)]
    total = sum(values)
    if total <= 0:
        print(f"    {label} cannot be all zero - keeping defaults.")
        return tuple(defaults)
    if abs(total - 100) > 0.01:
        print(f"    {label} sums to {total:g}% - normalising to 100%.")
        values = [v / total * 100 for v in values]
    return tuple(values)


In [16]:
# ---------------------------------------------------------------------------
# Interactive entry - covers every parameter
# ---------------------------------------------------------------------------

def collect_inputs(inp: Inputs = None) -> Inputs:
    inp = inp or Inputs()

    print("\n" + "=" * 70)
    print("Cement embodied carbon calculator - parameter entry")
    print("=" * 70)
    print("Enter a value, or press Enter (or type 'na') to use the default shown.")
    print("Defaults are literature-typical for a modern dry-process plant.")

    print("\n-- Cement composition ---------------------------------------------")
    inp.cement_type = ask_choice("Cement type", list(CEMENT_TYPES), inp.cement_type)
    preset = CEMENT_TYPES[inp.cement_type]
    inp.clinker_pct, inp.gypsum_pct, inp.scm_pct = ask_composition(
        "Cement composition",
        ["Clinker (%)", "Gypsum (%)", "SCM / addition (%)"],
        [preset["clinker"], preset["gypsum"], preset["scm"]])
    inp.scm_ef = (ask_float("SCM supply factor (kg CO2e per t SCM)",
                            preset["scm_ef"] or 79.0, 0, 1000)
                  if inp.scm_pct > 0 else 0.0)

    print("\n-- Clinker phase composition --------------------------------------")
    inp.phase_pct = ask_composition("Phase composition", PHASE_NAMES, inp.phase_pct)

    print("\n-- Raw materials --------------------------------------------------")
    inp.limestone_purity_pct = ask_float(
        "Limestone purity (% CaCO3)", inp.limestone_purity_pct, 1, 100)
    inp.clay_t_per_t_clinker = ask_float(
        "Clay (t per t clinker)", inp.clay_t_per_t_clinker, 0, 2)
    inp.sand_t_per_t_clinker = ask_float(
        "Sand (t per t clinker)", inp.sand_t_per_t_clinker, 0, 2)
    inp.iron_t_per_t_clinker = ask_float(
        "Iron ore (t per t clinker)", inp.iron_t_per_t_clinker, 0, 2)

    print("\n-- A1.1 Quarrying -------------------------------------------------")
    inp.quarry_diesel_l_per_t = ask_float(
        "Mobile plant diesel (L per t rock)", inp.quarry_diesel_l_per_t, 0, 20)
    inp.diesel_ef_per_l = ask_float(
        "Diesel emission factor (kg CO2/L)", inp.diesel_ef_per_l, 0, 5)
    inp.anfo_kg_per_t = ask_float(
        "Explosive (kg per t rock)", inp.anfo_kg_per_t, 0, 5)
    inp.anfo_ef = ask_float(
        "Explosive emission factor (kg CO2/kg)", inp.anfo_ef, 0, 5)
    inp.gypsum_ef = ask_float(
        "Gypsum supply factor (kg CO2e/t)", inp.gypsum_ef, 0, 500)

    print("\n-- A1.2 Crushing --------------------------------------------------")
    n_stages = ask_int("Number of crusher stages", len(inp.crushers), 1, 6)
    crushers = []
    for i in range(n_stages):
        throughput, power = inp.crushers[i] if i < len(inp.crushers) else (500.0, 200.0)
        print(f"  Stage {i + 1}:")
        crushers.append((ask_float("  Throughput (t/h)", throughput, 0.1, 5000),
                         ask_float("  Motor rating (kW)", power, 0.1, 5000)))
    inp.crushers = tuple(crushers)

    print("\n-- A2 Transport ---------------------------------------------------")
    for attr, label in (("raw_material_haul", "Raw materials quarry -> plant"),
                        ("gypsum_haul", "Gypsum supplier -> plant"),
                        ("scm_haul", "SCM supplier -> plant"),
                        ("fuel_haul", "Fuel supplier -> plant")):
        mode, distance = getattr(inp, attr)
        print(f"  {label}:")
        setattr(inp, attr, (ask_choice("  Mode", list(TRANSPORT_EF), mode),
                            ask_float("  Distance (km)", distance, 0, 20000)))

    print("\n-- A3.1 Kiln fuel -------------------------------------------------")
    inp.kiln_heat_mj_per_t_clinker = ask_float(
        "Specific heat consumption (MJ per t clinker)",
        inp.kiln_heat_mj_per_t_clinker, 1000, 10000)
    print("  Share of kiln heat by fuel (normalised to 100%):")
    shares = ask_composition("Fuel mix", list(FUELS),
                             [inp.fuel_mix.get(f, 0.0) for f in FUELS])
    inp.fuel_mix = {f: s for f, s in zip(FUELS, shares) if s > 0}

    print("\n-- A3.2 Electricity (kWh per t cement) ----------------------------")
    inp.power_raw_mill = ask_float("Raw mill", inp.power_raw_mill, 0, 500)
    inp.power_kiln_and_preheater = ask_float(
        "Kiln drive and preheater fans", inp.power_kiln_and_preheater, 0, 500)
    inp.power_cooler_fans = ask_float(
        "Clinker cooler fans", inp.power_cooler_fans, 0, 500)
    inp.power_cement_mill = ask_float(
        "Cement (finish) mill", inp.power_cement_mill, 0, 500)
    inp.power_conveyors_and_packing = ask_float(
        "Conveyors, pumps and packing", inp.power_conveyors_and_packing, 0, 500)

    print("\n-- Electricity grid -----------------------------------------------")
    inp.grid_ef = ask_float("Grid intensity (kg CO2e/kWh)", inp.grid_ef, 0, 2)

    return inp

In [17]:
def main() -> None:
    try:
        report(collect_inputs())
    except ValueError as error:
        print(f"\nerror: {error}")


if __name__ == "__main__":
    main()



Cement embodied carbon calculator - parameter entry
Enter a value, or press Enter (or type 'na') to use the default shown.
Defaults are literature-typical for a modern dry-process plant.

-- Cement composition ---------------------------------------------


  Cement type (CEM I, CEM II/B-S, CEM III/A) [CEM I]:  CEM I
  Clinker (%) [95]:  
  Gypsum (%) [5]:  
  SCM / addition (%) [0]:  



-- Clinker phase composition --------------------------------------


  C3S [52.6]:  
  C2S [26.3]:  
  C3A [10.6]:  
  C4AF [10.5]:  



-- Raw materials --------------------------------------------------


...


-- A3.2 Electricity (kWh per t cement) ----------------------------


  Raw mill [25]:  
  Kiln drive and preheater fans [25]:  
  Clinker cooler fans [5]:  
  Cement (finish) mill [40]:  
  Conveyors, pumps and packing [8]:  



-- Electricity grid -----------------------------------------------


  Grid intensity (kg CO2e/kWh) [0.233]:  0.3



CEM I - cradle to gate (A1-A3), 1 tonne of cement

Clinker phase composition
  C3S        52.6 %      CaO fraction 0.7368
  C2S        26.3 %      CaO fraction 0.6512
  C3A        10.6 %      CaO fraction 0.6226
  C4AF       10.5 %      CaO fraction 0.4616
  CaO in clinker  67.33 %

Material balance (tonne per tonne of cement)
  caco3                                           1.142
  limestone                                       1.202
  clay                                            0.190
  sand                                            0.028
  iron ore                                        0.019
  gypsum                                          0.050

Emissions by line item (kg CO2e per tonne of cement)
  A1.1 quarrying - mobile plant diesel             3.86
  A1.1 quarrying - blasting (ANFO)                 0.13
  A1.2 crushing and screening                      0.32
  A1.3 gypsum supply                               0.75
  A1.3 SCM supply                                  0.00
